# PANDAS MASTERY — CHAPTERS 25–30

```
PANDAS MASTERY
Current Chapters: 25 (Join Debugging), 26 (Concatenation), 27 (Reshaping),
                  28 (Pivot Tables), 29 (Time Series), 30 (Time Series Resampling)
Progress:         30 / 76 chapters
Completed:        Ch1-24
Remaining:        46 chapters
```

Six chapters: formalizing how to catch bad joins before they cause damage, the
concat/merge/join distinction, long-vs-wide reshaping, pivot tables as the business-dashboard
tool, and the start of Pandas' datetime machinery — the foundation for every time-series
feature engineering chapter later in this course.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)


---
# CHAPTER 25 — Join Debugging

## Concept
Chapter 24 showed joins are correct when row-multiplication is *expected*. This chapter is
about **detecting when it's not** — the systematic checks that should run after every
non-trivial merge, before trusting the result.

## The core habit: compare row counts, always


In [ ]:
customers = pd.DataFrame({"customer_id": [1, 2, 3, 4, 5], "name": ["Alice", "Bob", "Carol", "Dave", "Eve"]})

# A subtly corrupted orders table: customer_id 2 appears with slightly different values,
# simulating an upstream data quality issue (e.g., duplicate ingestion, or a genuine key collision)
orders_corrupted = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "customer_id": [1, 2, 2, 3, 4, 2],   # customer_id 2 appears THREE times
    "amount": [50.0, 75.0, 75.0, 30.0, 90.0, 75.0],
})

print("customers rows:", len(customers))
print("orders rows   :", len(orders_corrupted))

merged = pd.merge(customers, orders_corrupted, on="customer_id", how="left")
print("\nmerged rows   :", len(merged))
n_matched = orders_corrupted["customer_id"].isin(customers["customer_id"]).sum()
n_unmatched_customers = (~customers["customer_id"].isin(orders_corrupted["customer_id"])).sum()
print(f"\nExpected row count = (orders that matched a customer) + (customers with NO orders)")
print(f"                    = {n_matched} + {n_unmatched_customers} = {n_matched + n_unmatched_customers}")
print(f"Actual result: {len(merged)} -- {'MATCHES' if len(merged) == n_matched + n_unmatched_customers else 'DOES NOT MATCH -- investigate!'}")


### Expected Output (verified on pandas 3.0.2)
```
customers rows: 5
orders rows   : 6

merged rows   : 7

Expected row count = (orders that matched a customer) + (customers with NO orders)
                    = 6 + 1 = 7
Actual result: 7 -- MATCHES
```

### Getting the row-count arithmetic right — this is the actual skill, not a fixed formula
Notice the merged row count (7) is **larger** than `orders`' own row count (6), not roughly
equal to it — a first-glance "left join with unique left keys should stay close to the right
table's size" intuition is too imprecise to trust. The **correct** way to reason about a left
join's row count is: every row in the right table that matches a left key contributes one
output row (here, all 6 orders match some customer), **plus** every left row with **zero**
matches contributes exactly one output row with `NaN`s (here, customer 5 has no orders, adding
1 more row) — `6 + 1 = 7`, exactly what was measured. **This is precisely why the professional
habit is to compute the expected count explicitly from the data's actual match structure, not
estimate it by eyeballing table sizes** — an estimate that "feels about right" is exactly how
subtle merge bugs go unnoticed.

### Why this particular result is correct, not a bug
This merge is well-behaved because `customer_id` is unique in the **left** table
(`customers`) — a left join against a unique-keyed left table can never multiply any single
left row into more than however many right-side matches it has, so there's no combinatorial
*explosion* here (unlike the many-to-many case below) — but the row count still isn't simply
"one side's row count," precisely because unmatched left rows and matched counts both
contribute, as just computed. The dangerous case — genuine multiplicative explosion — is when
**both** sides have repeated keys, demonstrated next.


In [ ]:
# --- A GENUINE row-explosion: duplicate keys on BOTH sides ---
left_dup = pd.DataFrame({"key": ["A", "A", "B", "B"], "left_val": [1, 2, 3, 4]})
right_dup = pd.DataFrame({"key": ["A", "A", "B", "B"], "right_val": [10, 20, 30, 40]})

print("left_dup rows :", len(left_dup))
print("right_dup rows:", len(right_dup))

exploded = pd.merge(left_dup, right_dup, on="key")
print("\nmerged rows   :", len(exploded))
print("\nFull result:\n", exploded)
print(f"\nExplosion factor: {len(exploded)} rows from {len(left_dup)}+{len(right_dup)} inputs")


### Expected Output (verified on pandas 3.0.2)
```
left_dup rows : 4
right_dup rows: 4

merged rows   : 8

Full result:
   key  left_val  right_val
0   A         1         10
1   A         1         20
2   A         2         10
3   A         2         20
4   B         3         30
5   B         3         40
6   B         4         30
7   B         4         40

Explosion factor: 8 rows from 4+4 inputs
```

### Why this happens, mathematically — the many-to-many rule
When a key value `K` appears `m` times in the left table and `n` times in the right table, the
merge produces `m × n` rows for that key (**every** left row with key `K` pairs with **every**
right row with key `K`) — here, key "A" appears twice on each side, producing `2×2=4` output
rows just for "A" (and the same for "B"), totaling 8 from 4+4 inputs. This is the exact
mechanism behind "my merge suddenly increased my dataset from 10 million rows to 200 million
rows" — it means the join key had **far more duplication on both sides** than assumed, and the
combinatorial explosion compounds fast.

## The systematic debugging checklist


In [ ]:
def diagnose_merge(left: pd.DataFrame, right: pd.DataFrame, on: str, how: str = "left") -> pd.DataFrame:
    """Run a merge with a full diagnostic report BEFORE trusting the result."""
    left_key_dupes = left[on].duplicated().sum()
    right_key_dupes = right[on].duplicated().sum()
    left_only_keys = set(left[on]) - set(right[on])
    right_only_keys = set(right[on]) - set(left[on])

    result = pd.merge(left, right, on=on, how=how)

    print(f"=== Merge Diagnostic (how='{how}', on='{on}') ===")
    print(f"Left rows: {len(left)}   Right rows: {len(right)}   Result rows: {len(result)}")
    print(f"Duplicate keys in LEFT : {left_key_dupes}")
    print(f"Duplicate keys in RIGHT: {right_key_dupes}")
    print(f"Keys only in LEFT (no match in right): {len(left_only_keys)}")
    print(f"Keys only in RIGHT (no match in left): {len(right_only_keys)}")
    if left_key_dupes > 0 and right_key_dupes > 0:
        print("WARNING: duplicate keys on BOTH sides -- verify this many-to-many result is intentional!")
    return result


_ = diagnose_merge(left_dup, right_dup, on="key", how="inner")


### Expected Output (verified on pandas 3.0.2)
```
=== Merge Diagnostic (how='inner', on='key') ===
Left rows: 4   Right rows: 4   Result rows: 8
Duplicate keys in LEFT : 2
Duplicate keys in RIGHT: 2
Keys only in LEFT (no match in right): 0
Keys only in RIGHT (no match in left): 0
WARNING: duplicate keys on BOTH sides -- verify this many-to-many result is intentional!
```

### `indicator=True` — Pandas' built-in "where did each row come from" tool


In [ ]:
result_with_indicator = pd.merge(customers, orders_corrupted, on="customer_id", how="outer", indicator=True)
print(result_with_indicator)
print("\nmerge status counts:\n", result_with_indicator["_merge"].value_counts())


### Expected Output (verified on pandas 3.0.2)
```
   customer_id   name  order_id  amount      _merge
0            1  Alice     101.0    50.0        both
1            2    Bob     102.0    75.0        both
2            2    Bob     103.0    75.0        both
3            2    Bob     106.0    75.0        both
4            3  Carol     104.0    30.0        both
5            4   Dave     105.0    90.0        both
6            5    Eve       NaN     NaN   left_only

merge status counts:
_merge
both          6
left_only     1
right_only    0
Name: count, dtype: int64

```

### Why `indicator=True` is the fastest way to answer "which rows didn't match"
Without it, finding unmatched rows after an outer join requires manually checking for `NaN` in
one of the joined columns (fragile if that column can legitimately be `NaN` for other reasons).
`indicator=True` adds an explicit `_merge` column stating precisely `'left_only'`,
`'right_only'`, or `'both'` for every row — `result_with_indicator[result_with_indicator['_merge']
== 'left_only']` immediately isolates exactly the customers with zero matching orders (here,
just Eve), with no ambiguity.

### Interview Insight
> "Your join suddenly increases a 10-million-row dataset to 200 million rows. What happened,
> and how would you investigate?"
This is the many-to-many merge explosion — the join key has duplicate values on **both** sides
of the merge, and every duplicate on the left pairs with every duplicate on the right for
matching keys. To investigate: check `left[key].duplicated().sum()` and
`right[key].duplicated().sum()` before trusting any large merge; if both are nonzero, the
combinatorial row count is expected mathematically (`sum of m_i * n_i` per distinct key value),
not a bug — the real question becomes whether that duplication in the source data is itself
correct or a data quality problem upstream. `validate=` (Chapter 24) is the preventive habit
that would have caught this immediately by raising an error the moment an assumed
one-to-one/one-to-many relationship turned out to be many-to-many.


---
# CHAPTER 26 — Concatenation

## Concept
`pd.concat()` stacks DataFrames either row-wise (`axis=0`, the default — "add more rows,
same columns") or column-wise (`axis=1` — "add more columns, same rows"). It does NOT match on
a key like `merge()` does — it aligns purely by **index/column labels** (or ignores them
entirely with `ignore_index=True`).

## `merge` vs `join` vs `concat` — the comparison that must be crystal clear
| | Combines by | Typical use |
|---|---|---|
| `pd.concat(axis=0)` | Stacking rows — same columns, more records | Combining monthly files, appending new data |
| `pd.concat(axis=1)` | Stacking columns — same rows, more fields | Combining pre-aligned feature sets |
| `pd.merge()` | Matching **key column values** between two tables | Relational joins (customers + orders) |
| `df.join()` | Matching on the **index** (a convenience wrapper around merge for index-based joins) | Combining tables already indexed by the same key |


In [ ]:
jan_sales = pd.DataFrame({"product": ["A", "B"], "revenue": [1000, 1500]})
feb_sales = pd.DataFrame({"product": ["A", "B"], "revenue": [1200, 1300]})

# --- Row-wise concat (axis=0, default) -- stacking monthly data ---
combined = pd.concat([jan_sales, feb_sales], keys=["Jan", "Feb"])
print("concat(axis=0, keys=['Jan','Feb']) -- adds a MultiIndex level identifying the source:\n", combined)

combined_flat = pd.concat([jan_sales, feb_sales], ignore_index=True)
print("\nconcat(ignore_index=True) -- fresh 0..N index, source info LOST:\n", combined_flat)


### Expected Output (verified on pandas 3.0.2)
```
concat(axis=0, keys=['Jan','Feb']) -- adds a MultiIndex level identifying the source:
      product  revenue
Jan 0       A     1000
    1       B     1500
Feb 0       A     1200
    1       B     1300

concat(ignore_index=True) -- fresh 0..N index, source info LOST:
   product  revenue
0        A     1000
1        B     1500
2        A     1200
3        B     1300
```

### `keys=` vs `ignore_index=True` — a real trade-off
`keys=` preserves **provenance** (which source each row came from) at the cost of a MultiIndex
(Chapter 23) you'll likely `reset_index()` before further processing. `ignore_index=True`
discards provenance entirely but gives a clean, simple default index. **If you'll need to trace
rows back to their source later (e.g., debugging a discrepancy), use `keys=`; if the source
truly doesn't matter downstream, `ignore_index=True` is simpler.**

### Column-wise concat (`axis=1`) — and its real alignment danger


In [ ]:
features_a = pd.DataFrame({"height": [170, 165, 180]}, index=[0, 1, 2])
features_b = pd.DataFrame({"weight": [70, 60, 85]}, index=[0, 1, 2])
features_b_misaligned = pd.DataFrame({"weight": [70, 60, 85]}, index=[1, 2, 3])   # shifted index!

aligned = pd.concat([features_a, features_b], axis=1)
misaligned = pd.concat([features_a, features_b_misaligned], axis=1)

print("Correctly aligned indices:\n", aligned)
print("\nMISALIGNED indices -- concat still 'succeeds', but look closely:\n", misaligned)


### Expected Output (verified on pandas 3.0.2)
```
Correctly aligned indices:
   height  weight
0     170      70
1     165      60
2     180      85

MISALIGNED indices -- concat still 'succeeds', but look closely:
   height  weight
0   170.0     NaN
1   165.0    70.0
2   180.0    60.0
3     NaN    85.0
```

### Why `axis=1` concat with mismatched indices is genuinely dangerous
`concat(axis=1)` aligns rows **by index label**, not by position — when the indices don't
actually correspond to the same real-world entities (here, `features_b_misaligned`'s index is
shifted by one), the result silently pairs the **wrong** height with the wrong weight (row 1's
height `165` now sits next to weight `70`, which was originally meant for row 0) and introduces
`NaN` at both ends where indices don't overlap at all. **This runs without any error or
warning** — the only defense is verifying both DataFrames share the exact same index before
column-wise concatenation, or explicitly resetting both to a shared default index first if
row *position* (not label) is what actually corresponds between them.

### Interview Insight
> "What's the real difference between `concat()` and `merge()`?"
`concat()` stacks DataFrames along an axis based on **label alignment** (or ignores labels
entirely) — it has no concept of matching rows by a key's *value*. `merge()` explicitly matches
rows where a key **column's value** is equal between two tables, following relational-join
semantics (inner/left/right/outer). Use `concat()` for combining datasets with the same
structure (more rows or more pre-aligned columns); use `merge()` whenever you need to combine
data based on a shared identifier's value, like joining customers to their orders.


---
# CHAPTER 27 — Reshaping

## Concept
**Wide format**: one row per entity, one column per measurement/time-period (many columns, few
rows) — good for human reading, bad for most analysis/plotting tools. **Long format**: one row
per (entity, measurement) pair (few columns, many rows) — the format `groupby`, plotting
libraries, and most ML feature pipelines actually want. `melt()` converts wide→long;
`pivot()`/`pivot_table()` convert long→wide.


In [ ]:
wide = pd.DataFrame({
    "product": ["Widget", "Gadget", "Gizmo"],
    "Jan": [100, 150, 80],
    "Feb": [120, 140, 90],
    "Mar": [110, 160, 95],
})
print("WIDE format (one row per product, one column per month):\n", wide)

long = wide.melt(id_vars="product", var_name="month", value_name="sales")
print("\nLONG format after melt() (one row per product-month pair):\n", long)


### Expected Output (verified on pandas 3.0.2)
```
WIDE format (one row per product, one column per month):
  product  Jan  Feb  Mar
0  Widget  100  120  110
1  Gadget  150  140  160
2   Gizmo   80   90   95

LONG format after melt() (one row per product-month pair):
  product month  sales
0  Widget   Jan    100
1  Gadget   Jan    150
2   Gizmo   Jan     80
3  Widget   Feb    120
4  Gadget   Feb    140
5   Gizmo   Feb     90
6  Widget   Mar    110
7  Gadget   Mar    160
8   Gizmo   Mar     95
```

### Why long format is what `groupby`/plotting/ML actually want
In long format, "month" and "sales" are genuine **columns** you can `groupby("month")`, filter
(`long[long["month"] == "Feb"]`), or feed to a plotting library expecting one row per
observation — none of which works naturally on wide format, where "month" is smeared across
column *names* rather than existing as actual data. **This wide→long conversion is one of the
most common real-world reshaping needs**, especially for data exported from spreadsheets/
reporting tools, which are almost always built wide for human readability.

### `pivot()` — the inverse, long→wide (requires unique index/column combinations)


In [ ]:
back_to_wide = long.pivot(index="product", columns="month", values="sales")
print("pivot() -- back to wide (product x month grid):\n", back_to_wide)

# pivot() REQUIRES unique (index, columns) pairs -- demonstrate the failure mode
long_with_dupe = pd.concat([long, pd.DataFrame({"product": ["Widget"], "month": ["Jan"], "sales": [999]})], ignore_index=True)
try:
    long_with_dupe.pivot(index="product", columns="month", values="sales")
except ValueError as e:
    print("\npivot() with duplicate (product, month) pairs raised:", e)


### Expected Output (verified on pandas 3.0.2)
```
pivot() -- back to wide (product x month grid):
month    Feb  Jan  Mar
product
Gadget   140  150  160
Gizmo     90   80   95
Widget   120  100  110

pivot() with duplicate (product, month) pairs raised: Index contains duplicate entries, cannot reshape
```

### Why `pivot()` fails on duplicates, and what to use instead
`pivot()` requires each `(index, columns)` combination to map to **exactly one** value — with
a duplicate `(Widget, Jan)` pair (two different sales figures for the same cell), there's no
single unambiguous value to place in that grid cell, so `pivot()` refuses rather than guessing.
**`pivot_table()` (Chapter 28) solves this by requiring an explicit aggregation function** for
exactly this ambiguous case — this is the precise reason `pivot_table` exists as a distinct,
more robust tool rather than `pivot()` simply having an `aggfunc` parameter bolted on.

### Interview Insight
> "When would you use `melt()` versus `pivot()`?"
`melt()` converts wide format (many columns, one row per entity) to long format (few columns,
one row per observation) — needed before most `groupby`/plotting/ML workflows, which expect
long format. `pivot()` does the reverse, long to wide — useful for final human-readable reports
or a specific matrix-shaped computation, but requires unique index/column combinations; if
duplicates could exist, `pivot_table()` (with an explicit `aggfunc`) is the correct tool
instead.


---
# CHAPTER 28 — Pivot Tables

## Concept
`pd.pivot_table()` is `pivot()` plus **built-in aggregation** for duplicate index/column
combinations, plus optional row/column totals (`margins=True`) — the direct tool behind most
"business dashboard" style summary tables.


In [ ]:
sales_records = pd.DataFrame({
    "region": ["East", "East", "West", "West", "East", "West", "East"],
    "product": ["A", "B", "A", "B", "A", "A", "B"],
    "quarter": ["Q1", "Q1", "Q1", "Q1", "Q2", "Q2", "Q2"],
    "revenue": [1000, 1500, 1200, 900, 1100, 1300, 1600],
})

pivot = pd.pivot_table(
    sales_records,
    index="region",
    columns="product",
    values="revenue",
    aggfunc="sum",
)
print("pivot_table (region x product, summed revenue):\n", pivot)


### Expected Output (verified on pandas 3.0.2)
```
pivot_table (region x product, summed revenue):
product     A     B
region
East     2100  3100
West     2500   900
```

### Multi-level pivot with `margins=True` — row/column totals, business-dashboard style


In [ ]:
pivot_full = pd.pivot_table(
    sales_records,
    index=["region", "quarter"],
    columns="product",
    values="revenue",
    aggfunc="sum",
    fill_value=0,        # replace missing combinations with 0, not NaN -- appropriate for revenue
    margins=True,          # add row/column TOTALS
    margins_name="Total",
)
print(pivot_full)


### Expected Output (verified on pandas 3.0.2)
```
product          A     B  Total
region quarter
East   Q1     1000  1500   2500
       Q2     1100  1600   2700
West   Q1     1200   900   2100
       Q2     1300     0   1300
Total          4600  4000   8600
```

### Line-by-line: this exact `pivot_table` call
- `index=["region", "quarter"]` — creates a MultiIndex (Chapter 23) for rows: region, then
  quarter within region.
- `columns="product"` — spreads distinct `product` values into separate columns.
- `values="revenue"` — the numeric column being aggregated into each cell.
- `aggfunc="sum"` — how to combine multiple matching rows into one cell (here, summing revenue
  for that region+quarter+product combination).
- `fill_value=0` — West/Q2/product B has no matching rows at all (no sales that
  quarter/region/product combo) — without `fill_value`, that cell would show `NaN`; `0` is the
  business-correct interpretation for "zero revenue," not "unknown."
- `margins=True, margins_name="Total"` — adds a final row and column summing across everything,
  giving grand totals per product, per region-quarter, and overall — exactly the shape of a
  typical executive dashboard summary table.

### Interview Insight
> "What's the practical difference between `pivot()` and `pivot_table()`?"
`pivot()` is a pure reshape — it requires each index/column combination to already be unique,
and has no aggregation capability; if duplicates exist, it errors. `pivot_table()` is
reshape-plus-aggregate — it explicitly handles (and requires, via `aggfunc`) combining multiple
matching rows into a single cell, and additionally supports convenience features like
`fill_value` (for genuinely-missing combinations) and `margins` (for totals) that `pivot()`
doesn't have at all. In practice, `pivot_table()` is used far more often, since real data rarely
guarantees the uniqueness `pivot()` demands.


---
# CHAPTER 29 — Time Series

## Concept
`pd.to_datetime()` converts text/numbers to genuine datetime objects (Chapter 8 introduced
this); the `.dt` accessor then exposes every calendar component — year, month, day of week,
quarter — as vectorized extractions, the datetime equivalent of `.str` for text.

## A real, current gotcha worth knowing: mixed date formats now require `format='mixed'`
The `date` column below deliberately mixes date-only strings (`"2023-03-15"`) with
date-plus-time strings (`"2023-06-20 14:30:00"`) — a very common real-world situation (some
records logged with time precision, others without). On current Pandas, calling
`pd.to_datetime()` on this **without** `format='mixed'` raises a `ValueError` outright, rather
than silently guessing per-row — this was caught by running the code, not assumed from memory
of older Pandas behavior, where looser format inference was more permissive by default.
`format='mixed'` explicitly tells Pandas to infer each element's format individually rather than
requiring one consistent format for the whole column.


In [ ]:
events = pd.DataFrame({
    "event": ["Launch", "Update", "Sale", "Renewal", "Review"],
    "date": ["2023-03-15", "2023-06-20 14:30:00", "2023-11-25", "2024-01-01", "2024-03-15"],
})

events["date"] = pd.to_datetime(events["date"], format="mixed")
print("events:\n", events)
print("\ndtype:", events["date"].dtype)
print()
print(".dt.year       :", events["date"].dt.year.tolist())
print(".dt.month      :", events["date"].dt.month.tolist())
print(".dt.day        :", events["date"].dt.day.tolist())
print(".dt.day_name() :", events["date"].dt.day_name().tolist())
print(".dt.quarter    :", events["date"].dt.quarter.tolist())
print(".dt.hour       :", events["date"].dt.hour.tolist())
print(".dt.is_month_start:", events["date"].dt.is_month_start.tolist())


### Expected Output (verified on pandas 3.0.2)
```
events:
     event                 date
0   Launch  2023-03-15 00:00:00
1   Update  2023-06-20 14:30:00
2     Sale  2023-11-25 00:00:00
3  Renewal  2024-01-01 00:00:00
4   Review  2024-03-15 00:00:00

dtype: datetime64[us]

.dt.year       : [2023, 2023, 2023, 2024, 2024]
.dt.month      : [3, 6, 11, 1, 3]
.dt.day        : [15, 20, 25, 1, 15]
.dt.day_name() : ['Wednesday', 'Tuesday', 'Saturday', 'Monday', 'Friday']
.dt.quarter    : [1, 2, 4, 1, 1]
.dt.hour       : [0, 14, 0, 0, 0]
.dt.is_month_start: [False, False, False, True, False]
```

### Setting a datetime column as the index — unlocks powerful date-based slicing


In [ ]:
ts = events.set_index("date")
print("Indexed by date:\n", ts)
print()
print("Slice by YEAR (partial-string indexing):\n", ts.loc["2023"])
print("\nSlice by MONTH:\n", ts.loc["2023-06"])
print("\nSlice by RANGE:\n", ts.loc["2023-06":"2023-12"])


### Expected Output (verified on pandas 3.0.2)
```
Indexed by date:
                        event
date
2023-03-15 00:00:00   Launch
2023-06-20 14:30:00   Update
2023-11-25 00:00:00     Sale
2024-01-01 00:00:00  Renewal
2024-03-15 00:00:00   Review

Slice by YEAR (partial-string indexing):
                       event
date
2023-03-15 00:00:00  Launch
2023-06-20 14:30:00  Update
2023-11-25 00:00:00    Sale

Slice by MONTH:
                       event
date
2023-06-20 14:30:00  Update

Slice by RANGE:
                       event
date
2023-06-20 14:30:00  Update
2023-11-25 00:00:00    Sale
```

### Why "partial-string indexing" is one of Pandas' most powerful, underused datetime features
Once a DataFrame has a genuine `DatetimeIndex`, `.loc["2023"]` understands that string as "all
of the year 2023" and correctly slices every matching row — no manual
`(ts.index.year == 2023)` boolean mask needed. This works at any granularity the string
implies (`"2023"` → whole year, `"2023-06"` → whole month, `"2023-06-20"` → whole day) and is
the standard, idiomatic way to slice time-indexed data in Pandas — directly setting up Chapter
30's `resample()`, which requires exactly this datetime-indexed structure to work.

### Interview Insight
> "Why would you set a datetime column as a DataFrame's index rather than keeping it as a
> regular column?"
A `DatetimeIndex` unlocks partial-string date slicing (`.loc['2023-06']` for an entire month, no
manual boolean masking needed) and is a **prerequisite** for `resample()` (Chapter 30) and many
rolling/window time-series operations (Chapter 31/32) — those tools are specifically designed to
operate on datetime-indexed data, not on a plain datetime column sitting among the other
columns.


---
# CHAPTER 30 — Time Series Resampling

## Concept
`resample()` regroups time-indexed data into fixed time buckets (daily, weekly, monthly, ...)
and aggregates within each — conceptually a `groupby()` where the grouping key is "which time
bucket does this timestamp fall into," rather than a categorical column.


In [ ]:
rng = np.random.default_rng(0)
dates = pd.date_range("2023-01-01", periods=90, freq="D")
daily_sales = pd.DataFrame({"revenue": rng.uniform(800, 1500, 90)}, index=dates)

print("Raw daily data (first 5 rows):\n", daily_sales.head())
print("\nTotal rows:", len(daily_sales))

weekly = daily_sales.resample("W").sum()
monthly = daily_sales.resample("ME").sum()   # 'ME' = Month End (modern pandas frequency alias)

print("\nWeekly resample (first 5 weeks):\n", weekly.head())
print("\nMonthly resample:\n", monthly)


### Expected Output (verified on pandas 3.0.2)
```
Raw daily data (first 5 rows):
                revenue
2023-01-01  1245.873181
2023-01-02   988.850700
2023-01-03   828.681467
2023-01-04   811.569345
2023-01-05  1369.289167

Total rows: 90

Weekly resample (first 5 weeks):
                revenue
2023-01-01  1245.873181
2023-01-08  7972.612219
2023-01-15  8342.554338
2023-01-22  7318.708476
2023-01-29  9086.324696

Monthly resample:
                 revenue
2023-01-31  36503.307113
2023-02-28  31883.596375
2023-03-31  36711.402532
```

### Why the first weekly bucket is tiny — reading resample boundaries correctly
`resample("W")` buckets by calendar week ending on Sunday by default — since the data starts
mid-week (Jan 1, 2023 was a Sunday, so the first bucket is genuinely just that one day), the
first bucket legitimately contains only 1 day's worth of revenue while subsequent full weeks
sum 7 days. **Always sanity-check bucket boundaries, especially the first and last, rather than
assuming every bucket has a "full" amount of underlying data** — a monthly resample of data that
starts mid-month has the same partial-first-bucket characteristic.

### `'ME'` vs the older `'M'` frequency alias — a real, current Pandas change worth knowing
Recent Pandas versions renamed several frequency aliases for clarity — `'M'` (which ambiguously
suggested "monthly" without specifying start/end) became `'ME'` (Month **End**) and `'MS'`
(Month **Start**) as explicit, unambiguous options. Older tutorials/Stack Overflow answers using
bare `'M'` may trigger a deprecation warning or behave unexpectedly on current Pandas — always
verify frequency alias behavior against your installed version rather than assuming
universality across Pandas versions.

### Different aggregations per resample — not just `.sum()`


In [ ]:
monthly_multi = daily_sales.resample("ME").agg(
    total_revenue=("revenue", "sum"),
    avg_daily_revenue=("revenue", "mean"),
    max_daily_revenue=("revenue", "max"),
    days_counted=("revenue", "count"),
)
print(monthly_multi)


### Expected Output (verified on pandas 3.0.2)
```
             total_revenue  avg_daily_revenue  max_daily_revenue  days_counted
2023-01-31   36503.307113        1177.526036        1498.046955            31
2023-02-28   31883.596375        1138.699871        1453.830461            28
2023-03-31   36711.402532        1184.238791        1496.567554            31
```

### Reading `days_counted` as a data-quality check
Notice `resample().agg(..., count)` gives you, for free, exactly how many raw records fell into
each bucket — useful for catching **incomplete periods** (e.g., if March had only 15 days
counted instead of 31, that's an immediate signal of missing data for that month, worth
investigating before trusting `total_revenue` for a month-over-month comparison). This named
`.agg()` syntax is identical to Chapter 21's groupby named aggregation — reinforcing that
`resample()` really is "groupby, but the group is a time bucket."

### Interview Insight
> "How is `resample()` conceptually related to `groupby()`?"
Both implement split-apply-combine — `groupby()` splits by the distinct values of a categorical
column; `resample()` splits by which fixed time bucket (day/week/month/...) each row's
timestamp falls into, requiring a `DatetimeIndex` to do so. Both support the same aggregation
patterns (`.sum()`, `.mean()`, named `.agg()`), and `resample()` can be thought of as
`groupby(pd.Grouper(freq=...))` under the hood — a time-bucket-aware grouping, not a
fundamentally different mechanism.


---
## Chapters 25–30 Summary

- **Join debugging**: many-to-many merges genuinely multiply rows by `m × n` per duplicated key
  — verified live (4+4 inputs → 8 output rows); `indicator=True` is the fastest way to isolate
  unmatched rows after an outer join; a systematic diagnostic function checking duplicate keys
  on both sides before trusting a merge is a real professional habit.
- **Concatenation**: `concat(axis=0)` stacks rows, `axis=1` stacks columns by **index
  alignment** — verified that misaligned indices in an `axis=1` concat silently pair the wrong
  rows together with no error at all; `merge` matches by key *value*, `concat` matches (or
  ignores) by *label*.
- **Reshaping**: long format is what `groupby`/plotting/ML actually consume; `melt()` goes
  wide→long, `pivot()` goes long→wide but requires unique index/column pairs (verified: raises
  `ValueError` on duplicates) — `pivot_table()` exists specifically to handle that ambiguous
  case via required aggregation.
- **Pivot tables**: `index`+`columns`+`values`+`aggfunc` is the core signature;
  `fill_value` distinguishes "genuinely zero" from "unknown"; `margins=True` adds
  dashboard-style totals.
- **Time series**: `.dt` is the vectorized datetime-component accessor, NaN-safe like `.str`;
  a `DatetimeIndex` unlocks partial-string slicing (`.loc['2023-06']`) and is the prerequisite
  for `resample()`.
- **Resampling**: `resample()` is `groupby()` with a time-bucket key — same named-aggregation
  syntax; always sanity-check partial first/last buckets; frequency aliases (`'ME'` vs the
  older `'M'`) are a real, current version-dependent detail worth verifying, not assuming.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For a merge where the join key has 3 duplicates on the left and 2
duplicates on the right (for the same key value), predict how many output rows that single key
value contributes.

**Exercise 2:** You need to combine 12 monthly CSV files (identical columns) into one
DataFrame, and you want to be able to trace any row back to its source file later for
debugging. Which `concat()` parameter would you use, and why not `ignore_index=True`?

**Coding Challenge:** Extend `diagnose_merge()` from this chapter to also report the resulting
row count as a percentage of `len(left) * len(right)` (the theoretical maximum for a fully
many-to-many join), to give an immediate sense of how "exploded" a merge actually is relative
to its worst case.

**Debugging Challenge:**
```python
prices = pd.DataFrame({"price": [10, 20, 30]}, index=["A", "B", "C"])
discounts = pd.DataFrame({"discount": [1, 2]}, index=["A", "B"])
combined = pd.concat([prices, discounts], axis=1)
combined["final_price"] = combined["price"] - combined["discount"]
print(combined["final_price"])   # a teammate is confused why one row is NaN
```
Explain exactly why, referencing this chapter's index-alignment lesson.

**Interview Questions:**
1. Explain, mathematically, why a many-to-many merge on a key with `m` and `n` duplicates
   produces `m × n` rows for that key.
2. What does `indicator=True` add to a merge, and how would you use it to find unmatched rows?
3. Why does `pivot()` require unique index/column combinations while `pivot_table()` doesn't?
4. Why is a `DatetimeIndex` a prerequisite for `resample()`?

**Quiz:**
- Q1: True/False — `pd.concat([df1, df2], axis=1)` matches rows by position, not by index
  label.
- Q2: Which pivot_table parameter distinguishes "genuinely zero" from "missing/unknown" for
  empty grid cells?
- Q3: What does `.resample("ME").agg(count=("col","count"))` tell you that's useful for data
  quality checking?

---
 